# 01 · Data exploration and preparation
WBS 1.2 and 1.3: load sync records, inspect normal behaviour, clean, transform. Run `python run_all.py` once first, and start Jupyter from the project root.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..')); os.chdir('..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import config, prepare_data
pd.set_option('display.width', 140)

## 1.2.1 Load sample synchronization records

In [ ]:
raw = pd.read_csv(config.RAW_DATA_FILE)
print(raw.shape)
raw.head()

## 1.2.2 Analyze synchronization features
Product, channel, price, stock, update time, sync delay, sync attempts, sync status.

In [ ]:
raw.describe(include='all').T

In [ ]:
raw.isna().sum()

## 1.3.1 Preprocessing
Duplicates, gaps, impossible values and inconsistent status labels are fixed and counted.

In [ ]:
clean, report = prepare_data.clean(raw)
report

## 1.2.3 Normal sync patterns by channel

In [ ]:
prepare_data.channel_stats(clean)

In [ ]:
normal = clean[clean.is_anomaly == 0]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
normal.assign(r=normal.channel_price / normal.master_price).boxplot(column='r', by='channel', ax=ax[0]); ax[0].set_ylim(.9, 1.1); ax[0].set_title('Price ratio to master')
normal.sync_delay_min.clip(upper=60).hist(bins=60, ax=ax[1]); ax[1].set_title('Minutes since last sync')
normal.sample(3000, random_state=1).plot.scatter(x='master_stock', y='channel_stock', alpha=.3, ax=ax[2]); ax[2].set_title('Channel vs master stock')
plt.suptitle(''); plt.tight_layout(); plt.show()

## 1.3.2 Transform categorical and timestamp attributes
Channel codes and one-hot columns, status code, cyclical hour and weekday.

In [ ]:
enc = prepare_data.encode(clean)
enc.head()

## 1.3.3 Normalize numerical features
The engineered features are standardized so no variable dominates the autoencoder.

In [ ]:
from features import build_features
feats, _ = build_features(clean)
feats.describe().T[['mean', 'std', 'min', 'max']].round(2)